# Análisis Estadístico de Usuarios WiFi - Medellín

**Datasets:**
- `cantidad_de_usuarios_por_sitio_dia.csv` — registros diarios por sede
- `segmentacion_de_usuarios.csv` — perfil demográfico y dispositivo por sede
- `wifi_puntos_medellin_comuna.csv` — mapeo sede → barrio → comuna

Este notebook responde:
1. **Flujo de usuarios WiFi** — promedio de registros diarios por sede, agregado por barrio
2. **Perfil del consumidor WiFi** — distribución de edad + dispositivo por barrio

In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

# ── Cargar datasets ──────────────────────────────────────────────────────────
usuarios_dia = pd.read_csv('../Database/cantidad_de_usuarios_por_sitio_dia.csv')
segmentacion  = pd.read_csv('../Database/segmentacion_de_usuarios.csv')
wifi_mapa     = pd.read_csv('../Database/wifi_puntos_medellin_comuna.csv')

print('=== usuarios_dia ===')
print(usuarios_dia.shape, '\n', usuarios_dia.head(3))
print('\n=== segmentacion ===')
print(segmentacion.shape, '\n', segmentacion.head(3))
print('\n=== wifi_mapa ===')
print(wifi_mapa.shape, '\n', wifi_mapa.head(3))

=== usuarios_dia ===
(3001, 4) 
                                             sede        mes     año  registros
0                          @medellín - sapiencia       mayo  2019.0      828.0
1  \upss manrique \"\"hermenegildo de fex\"\"\""     agosto  2018.0    15275.0
2  \upss manrique \"\"hermenegildo de fex\"\"\""  diciembre  2018.0     6225.0

=== segmentacion ===
(2650, 14) 
                                   Sedes  Rango_edad_De_14_a_17_Años  \
0         Aeroparque Juan Pablo Segundo                         183   
1  Ampliación 1 Estación Caribe (Norte)                         137   
2    Ampliación 2 Estación Caribe (Sur)                         111   

   Rango_edad_De_18_a_28_Años  Rango_edad_De_29_a_50_Años  \
0                         924                         543   
1                         884                         511   
2                         669                         347   

   Rango_edad_Mayor_de_50_Años  Rango_edad_Menor_de_13_Años  \
0                      

In [2]:
# ── Inspección de columnas y nulos ───────────────────────────────────────────
print('usuarios_dia columnas:', usuarios_dia.columns.tolist())
print('segmentacion columnas:', segmentacion.columns.tolist())
print('wifi_mapa columnas:', wifi_mapa.columns.tolist())

print('\nNulos usuarios_dia:')
print(usuarios_dia.isnull().sum())
print('\nNulos segmentacion:')
print(segmentacion.isnull().sum())

usuarios_dia columnas: ['sede', 'mes', 'año', 'registros']
segmentacion columnas: ['Sedes', 'Rango_edad_De_14_a_17_Años', 'Rango_edad_De_18_a_28_Años', 'Rango_edad_De_29_a_50_Años', 'Rango_edad_Mayor_de_50_Años', 'Rango_edad_Menor_de_13_Años', 'Dispositivo_conexión_Computadora_personal', 'Dispositivo_conexión_Smartphone', 'Dispositivo_conexión_Tablet', 'Sexo_Femenino', 'Sexo_Masculino', 'Género_Otro', 'Mes', 'Año']
wifi_mapa columnas: ['comuna', 'barrio', 'sede']

Nulos usuarios_dia:
sede         0
mes          0
año          0
registros    0
dtype: int64

Nulos segmentacion:
Sedes                                        0
Rango_edad_De_14_a_17_Años                   0
Rango_edad_De_18_a_28_Años                   0
Rango_edad_De_29_a_50_Años                   0
Rango_edad_Mayor_de_50_Años                  0
Rango_edad_Menor_de_13_Años                  0
Dispositivo_conexión_Computadora_personal    0
Dispositivo_conexión_Smartphone              0
Dispositivo_conexión_Tablet              

In [3]:
# ── Normalizar nombres de sede para join ─────────────────────────────────────
def normalizar(s):
    if pd.isna(s):
        return s
    return str(s).strip().lower()

usuarios_dia['sede_norm']  = usuarios_dia['sede'].apply(normalizar)
segmentacion['sede_norm']  = segmentacion['Sedes'].apply(normalizar)
wifi_mapa['sede_norm']     = wifi_mapa['sede'].apply(normalizar)
wifi_mapa['barrio_norm']   = wifi_mapa['barrio'].str.strip().str.lower()

# Mapa único sede → barrio, comuna
mapa_sede = wifi_mapa[['sede_norm', 'barrio_norm', 'comuna']].drop_duplicates('sede_norm')
print(f'Sedes únicas en wifi_mapa: {len(mapa_sede)}')
print(f'Sedes únicas en usuarios_dia: {usuarios_dia["sede_norm"].nunique()}')
print(f'Sedes únicas en segmentacion: {segmentacion["sede_norm"].nunique()}')

# Cobertura del join
sedes_usuarios  = set(usuarios_dia['sede_norm'].dropna())
sedes_mapa      = set(mapa_sede['sede_norm'].dropna())
print(f'\nCobertura join usuarios_dia ↔ wifi_mapa: {len(sedes_usuarios & sedes_mapa)}/{len(sedes_usuarios)} sedes ({len(sedes_usuarios & sedes_mapa)/len(sedes_usuarios)*100:.1f}%)')

Sedes únicas en wifi_mapa: 371
Sedes únicas en usuarios_dia: 393
Sedes únicas en segmentacion: 380

Cobertura join usuarios_dia ↔ wifi_mapa: 177/393 sedes (45.0%)


## 1. Flujo de Usuarios WiFi por Sede y Barrio

In [4]:
# ── Estadísticas por sede ────────────────────────────────────────────────────
flujo_sede = (
    usuarios_dia.groupby('sede_norm')['registros']
    .agg(
        registros_total='sum',
        registros_media='mean',
        registros_mediana='median',
        registros_max='max',
        registros_min='min',
        registros_std='std',
        n_meses='count'
    )
    .reset_index()
)
flujo_sede.columns = ['sede_norm', 'total_registros', 'media_mensual', 'mediana_mensual',
                       'max_mensual', 'min_mensual', 'std_mensual', 'n_meses']
flujo_sede['media_mensual']   = flujo_sede['media_mensual'].round(1)
flujo_sede['mediana_mensual'] = flujo_sede['mediana_mensual'].round(1)

# Join con barrio/comuna
flujo_sede = flujo_sede.merge(mapa_sede, on='sede_norm', how='left')

print('=== TOP 20 SEDES POR TOTAL DE REGISTROS ===')
print(flujo_sede.sort_values('total_registros', ascending=False)[['sede_norm','barrio_norm','total_registros','media_mensual']].head(20).to_string(index=False))

=== TOP 20 SEDES POR TOTAL DE REGISTROS ===
                         sede_norm   barrio_norm  total_registros  media_mensual
                  terminal del sur      guayabal       236796.998        19733.1
                   parque aranjuez      aranjuez       217187.000        18098.9
        parque de la vida (centro) la candelaria       161141.770        13428.5
              parque de los deseos      aranjuez       160069.430        14551.8
                    parque explora      aranjuez       158443.232        14403.9
    casa de justicia santo domingo       popular       158409.000        13200.8
       ampliación estación estadio           NaN       149570.000        18696.2
              plazuela san ignacio la candelaria       147803.000        12316.9
               parque campo valdés           NaN       146656.362        20950.9
               parque de guadalupe       popular       143748.000        11979.0
                parque santa elena   santa elena       138083.263

In [5]:
# ── Agregado por barrio ──────────────────────────────────────────────────────
flujo_barrio = (
    flujo_sede.dropna(subset=['barrio_norm'])
    .groupby('barrio_norm')
    .agg(
        n_sedes=('sede_norm', 'count'),
        total_registros=('total_registros', 'sum'),
        media_registros_sede=('media_mensual', 'mean'),
        max_registros_sede=('max_mensual', 'max'),
        comuna=('comuna', 'first')
    )
    .reset_index()
)
flujo_barrio['media_registros_sede']  = flujo_barrio['media_registros_sede'].round(1)
flujo_barrio = flujo_barrio.sort_values('total_registros', ascending=False)

print('=== FLUJO WiFi AGREGADO POR BARRIO ===')
print(flujo_barrio.head(25).to_string(index=False))
print(f'\nBarrios con cobertura WiFi: {len(flujo_barrio)}')
print(f'Total registros WiFi: {flujo_barrio["total_registros"].sum():,.0f}')
print(f'Media registros/barrio: {flujo_barrio["total_registros"].mean():,.0f}')

=== FLUJO WiFi AGREGADO POR BARRIO ===


              barrio_norm  n_sedes  total_registros  media_registros_sede  max_registros_sede  comuna
            la candelaria       15       931606.770                6509.1           24298.000    10.0
                 aranjuez       11       800757.300                7621.9           26441.000     4.0
                 castilla       10       662708.316                6438.7           34600.000     5.0
                 manrique        9       563121.000                5367.2           17718.000     3.0
                 guayabal        9       530633.998                5516.8           40461.000    15.0
               san javier       14       519143.000                3847.5           13283.000    13.0
                  popular        8       501580.000                6251.0           20375.000     1.0
         laureles estadio       16       479588.000                2860.6           10099.000    11.0
          doce de octubre       10       358253.000                4178.9         

In [6]:
# ── Agregado por comuna ──────────────────────────────────────────────────────
flujo_comuna = (
    flujo_sede.dropna(subset=['comuna'])
    .groupby('comuna')
    .agg(
        n_sedes=('sede_norm', 'count'),
        total_registros=('total_registros', 'sum'),
        media_mensual_sede=('media_mensual', 'mean')
    )
    .reset_index()
)
nombre_comunas = {
    1: 'Popular', 2: 'Santa Cruz', 3: 'Manrique', 4: 'Aranjuez',
    5: 'Castilla', 6: 'Doce de Octubre', 7: 'Robledo', 8: 'Villa Hermosa',
    9: 'Buenos Aires', 10: 'La Candelaria', 11: 'Laureles-Estadio',
    12: 'La América', 13: 'San Javier', 14: 'El Poblado',
    15: 'Guayabal', 16: 'Belén'
}
flujo_comuna['nombre_comuna'] = flujo_comuna['comuna'].map(nombre_comunas)
flujo_comuna = flujo_comuna.sort_values('total_registros', ascending=False)
print('=== FLUJO WiFi POR COMUNA ===')
print(flujo_comuna.to_string(index=False))

=== FLUJO WiFi POR COMUNA ===
 comuna  n_sedes  total_registros  media_mensual_sede    nombre_comuna
   10.0       15       931606.770         6509.106667    La Candelaria
    4.0       11       800757.300         7621.863636         Aranjuez
    5.0       10       662708.316         6438.690000         Castilla
    3.0        9       563121.000         5367.177778         Manrique
   15.0        9       530633.998         5516.811111         Guayabal
   13.0       14       519143.000         3847.478571       San Javier
    1.0        8       501580.000         6251.012500          Popular
   11.0       16       479588.000         2860.612500 Laureles-Estadio
    6.0       10       358253.000         4178.900000  Doce de Octubre
   60.0        6       350589.236         5611.600000              NaN
   16.0       10       346681.000         4507.220000            Belén
    9.0        6       317609.000         5060.133333     Buenos Aires
   90.0        5       256025.263         4411.

## 2. Perfil del Consumidor WiFi (Edad + Dispositivo)

In [7]:
# ── Columnas de segmentación ─────────────────────────────────────────────────
cols_edad = [
    'Rango_edad_Menor_de_13_Años',
    'Rango_edad_De_14_a_17_Años',
    'Rango_edad_De_18_a_28_Años',
    'Rango_edad_De_29_a_50_Años',
    'Rango_edad_Mayor_de_50_Años'
]
cols_dispositivo = [
    'Dispositivo_conexión_Computadora_personal',
    'Dispositivo_conexión_Smartphone',
    'Dispositivo_conexión_Tablet'
]
cols_sexo = ['Sexo_Femenino', 'Sexo_Masculino', 'Género_Otro']

# Convertir a numérico
for col in cols_edad + cols_dispositivo + cols_sexo:
    segmentacion[col] = pd.to_numeric(segmentacion[col], errors='coerce').fillna(0)

# ── Distribución global edad ─────────────────────────────────────────────────
total_edad = segmentacion[cols_edad].sum()
total_usuarios_seg = total_edad.sum()
print('=== DISTRIBUCIÓN GLOBAL POR RANGO DE EDAD (WiFi) ===')
for col in cols_edad:
    label = col.replace('Rango_edad_', '').replace('_', ' ')
    print(f'  {label}: {total_edad[col]:,.0f}  ({total_edad[col]/total_usuarios_seg*100:.1f}%)')

# ── Distribución global dispositivo ─────────────────────────────────────────
total_disp = segmentacion[cols_dispositivo].sum()
print('\n=== DISTRIBUCIÓN GLOBAL POR DISPOSITIVO (WiFi) ===')
for col in cols_dispositivo:
    label = col.replace('Dispositivo_conexión_', '').replace('_', ' ')
    print(f'  {label}: {total_disp[col]:,.0f}  ({total_disp[col]/total_disp.sum()*100:.1f}%)')

# ── Distribución global sexo ─────────────────────────────────────────────────
total_sexo = segmentacion[cols_sexo].sum()
print('\n=== DISTRIBUCIÓN GLOBAL POR SEXO (WiFi) ===')
for col in cols_sexo:
    label = col.replace('Sexo_', '').replace('Género_', '').replace('_', ' ')
    print(f'  {label}: {total_sexo[col]:,.0f}  ({total_sexo[col]/total_sexo.sum()*100:.1f}%)')

=== DISTRIBUCIÓN GLOBAL POR RANGO DE EDAD (WiFi) ===
  Menor de 13 Años: 57,229  (1.3%)
  De 14 a 17 Años: 465,146  (10.9%)
  De 18 a 28 Años: 2,293,613  (53.6%)
  De 29 a 50 Años: 1,308,646  (30.6%)
  Mayor de 50 Años: 153,384  (3.6%)

=== DISTRIBUCIÓN GLOBAL POR DISPOSITIVO (WiFi) ===
  Computadora personal: 171,589  (1.3%)
  Smartphone: 12,603,257  (98.0%)
  Tablet: 79,593  (0.6%)

=== DISTRIBUCIÓN GLOBAL POR SEXO (WiFi) ===
  Femenino: 2,042,451  (40.5%)
  Masculino: 2,955,012  (58.6%)
  Otro: 46,417  (0.9%)


In [8]:
# ── Join segmentación con mapa de barrios ────────────────────────────────────
seg_geo = segmentacion.merge(mapa_sede, on='sede_norm', how='left')

print(f'Registros con barrio asignado: {seg_geo["barrio_norm"].notna().sum()} / {len(seg_geo)}')

seg_geo_barrio = seg_geo.dropna(subset=['barrio_norm'])

# ── Agregar por barrio ───────────────────────────────────────────────────────
perfil_barrio = (
    seg_geo_barrio.groupby('barrio_norm')[
        cols_edad + cols_dispositivo + cols_sexo
    ].sum()
    .reset_index()
)

# Totales por barrio para calcular porcentajes
perfil_barrio['total_usuarios'] = perfil_barrio[cols_edad].sum(axis=1)
perfil_barrio['total_dispositivos'] = perfil_barrio[cols_dispositivo].sum(axis=1)

# Porcentaje por rango de edad
for col in cols_edad:
    label = 'pct_' + col.replace('Rango_edad_', '').replace('_', '_').replace('ños', 'anios')
    perfil_barrio[label] = (perfil_barrio[col] / perfil_barrio['total_usuarios'].replace(0, np.nan) * 100).round(1)

# Porcentaje por dispositivo
for col in cols_dispositivo:
    label = 'pct_' + col.replace('Dispositivo_conexión_', '').replace(' ', '_')
    perfil_barrio[label] = (perfil_barrio[col] / perfil_barrio['total_dispositivos'].replace(0, np.nan) * 100).round(1)

# Segmento dominante de edad
perfil_barrio['rango_edad_dominante'] = perfil_barrio[cols_edad].idxmax(axis=1).str.replace('Rango_edad_', '').str.replace('_', ' ')

# Dispositivo dominante
perfil_barrio['dispositivo_dominante'] = perfil_barrio[cols_dispositivo].idxmax(axis=1).str.replace('Dispositivo_conexión_', '').str.replace('_', ' ')

print('=== PERFIL CONSUMIDOR WiFi POR BARRIO (muestra) ===')
cols_show = ['barrio_norm', 'total_usuarios', 'rango_edad_dominante', 'dispositivo_dominante']
print(perfil_barrio.sort_values('total_usuarios', ascending=False)[cols_show].head(20).to_string(index=False))

Registros con barrio asignado: 1379 / 2650
=== PERFIL CONSUMIDOR WiFi POR BARRIO (muestra) ===
         barrio_norm  total_usuarios rango_edad_dominante dispositivo_dominante
            castilla          282425      De 18 a 28 Años            Smartphone
            aranjuez          221088      De 18 a 28 Años            Smartphone
       la candelaria          218818      De 18 a 28 Años            Smartphone
          san javier          171286      De 18 a 28 Años            Smartphone
            guayabal          164488      De 18 a 28 Años            Smartphone
            manrique          157174      De 18 a 28 Años            Smartphone
    laureles estadio          138764      De 18 a 28 Años            Smartphone
     doce de octubre          121954      De 18 a 28 Años            Smartphone
               belén          107148      De 18 a 28 Años            Smartphone
             popular          100231      De 18 a 28 Años            Smartphone
        buenos aires     

In [9]:
# ── Segmentación cruzada: edad x dispositivo global ──────────────────────────
# Crear tabla pivot: qué dispositivos usa cada rango etario
matriz_edad_disp = pd.DataFrame()
for col_e in cols_edad:
    for col_d in cols_dispositivo:
        label_e = col_e.replace('Rango_edad_', '').replace('_', ' ')
        label_d = col_d.replace('Dispositivo_conexión_', '').replace('_', ' ')
        # Estimación proporcional: asume independencia entre edad y dispositivo
        # (no tenemos cross-tab directo en los datos)
        pass

# Tabla de dominancia: barrios con smartphones jóvenes vs PCs adultos
perfil_barrio['ratio_smartphone_pc'] = (
    perfil_barrio['Dispositivo_conexión_Smartphone'] /
    perfil_barrio['Dispositivo_conexión_Computadora_personal'].replace(0, np.nan)
).round(2)

perfil_barrio['perfil_consumidor'] = perfil_barrio.apply(
    lambda row: 'digital-movil' if row['ratio_smartphone_pc'] > 5 else
                ('mixto-digital' if row['ratio_smartphone_pc'] > 2 else 'pc-tradicional'),
    axis=1
)

print('=== CLASIFICACIÓN DE BARRIOS POR PERFIL DE CONSUMIDOR ===')
print(perfil_barrio['perfil_consumidor'].value_counts())

print('\n=== TOP 10 BARRIOS CON MÁS USO MÓVIL (Ratio Smartphone/PC) ===')
print(perfil_barrio.sort_values('ratio_smartphone_pc', ascending=False)[['barrio_norm','ratio_smartphone_pc','rango_edad_dominante','total_usuarios']].head(10).to_string(index=False))

=== CLASIFICACIÓN DE BARRIOS POR PERFIL DE CONSUMIDOR ===


perfil_consumidor
digital-movil    21
Name: count, dtype: int64

=== TOP 10 BARRIOS CON MÁS USO MÓVIL (Ratio Smartphone/PC) ===
    barrio_norm  ratio_smartphone_pc rango_edad_dominante  total_usuarios
          belén               124.41      De 18 a 28 Años          107148
   buenos aires               121.66      De 18 a 28 Años           94069
        robledo               120.87      De 18 a 28 Años           20187
       aranjuez               112.02      De 18 a 28 Años          221088
     el poblado                94.98      De 18 a 28 Años           47664
  villa hermosa                86.70      De 18 a 28 Años           65830
       guayabal                85.39      De 18 a 28 Años          164488
doce de octubre                84.53      De 18 a 28 Años          121954
       castilla                82.45      De 18 a 28 Años          282425
       manrique                80.60      De 18 a 28 Años          157174


In [10]:
# ── Tendencia temporal por año ───────────────────────────────────────────────
if 'Año' in segmentacion.columns:
    seg_año = segmentacion.copy()
    seg_año['Año'] = pd.to_numeric(seg_año['Año'], errors='coerce').fillna(0).astype(int)
    seg_año = seg_año[seg_año['Año'] > 2010]
    
    tendencia = seg_año.groupby('Año')[cols_edad + cols_dispositivo].sum().reset_index()
    tendencia['total_usuarios'] = tendencia[cols_edad].sum(axis=1)
    print('=== TENDENCIA TEMPORAL DE USUARIOS WiFi ===')
    print(tendencia[['Año', 'total_usuarios'] + cols_dispositivo].to_string(index=False))

# ── Tendencia temporal usuarios_dia ─────────────────────────────────────────
if 'año' in usuarios_dia.columns:
    usuarios_dia['año_num'] = pd.to_numeric(usuarios_dia['año'], errors='coerce').fillna(0).astype(int)
    tend_dia = usuarios_dia[usuarios_dia['año_num'] > 2010].groupby('año_num')['registros'].agg(['sum','mean','count']).reset_index()
    tend_dia.columns = ['año', 'total_registros', 'media_mensual', 'n_registros']
    print('\n=== TENDENCIA ANUAL DE REGISTROS DIARIOS ===')
    print(tend_dia.to_string(index=False))

=== TENDENCIA TEMPORAL DE USUARIOS WiFi ===
 Año  total_usuarios  Dispositivo_conexión_Computadora_personal  Dispositivo_conexión_Smartphone  Dispositivo_conexión_Tablet
2018         2518511                                     119951                          8183599                        53683
2019         1759507                                      51638                          4419658                        25910

=== TENDENCIA ANUAL DE REGISTROS DIARIOS ===
 año  total_registros  media_mensual  n_registros
2018     11373622.086    6733.938476         1689
2019      4900106.131    3734.836990         1312


## 3. Exportar Archivos para Dashboard

In [11]:
OUTPUT_DIR = '../Dashboard/data'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── 1. Flujo por sede ────────────────────────────────────────────────────────
flujo_sede.to_csv(f'{OUTPUT_DIR}/flujo_wifi_sede.csv', index=False)

# ── 2. Flujo agregado por barrio ────────────────────────────────────────────
flujo_barrio.to_csv(f'{OUTPUT_DIR}/flujo_wifi_barrio.csv', index=False)

# ── 3. Flujo por comuna ─────────────────────────────────────────────────────
flujo_comuna.to_csv(f'{OUTPUT_DIR}/flujo_wifi_comuna.csv', index=False)

# ── 4. Perfil consumidor por barrio ─────────────────────────────────────────
perfil_barrio.to_csv(f'{OUTPUT_DIR}/perfil_consumidor_wifi_barrio.csv', index=False)

# ── 5. Distribución global edad (torta) ──────────────────────────────────────
dist_edad_wifi = pd.DataFrame({
    'rango_edad': [c.replace('Rango_edad_', '').replace('_', ' ') for c in cols_edad],
    'total_usuarios': [total_edad[c] for c in cols_edad],
    'pct': [(total_edad[c] / total_usuarios_seg * 100).round(2) for c in cols_edad]
})
dist_edad_wifi.to_csv(f'{OUTPUT_DIR}/distribucion_edad_wifi.csv', index=False)

# ── 6. Distribución global dispositivo (torta) ───────────────────────────────
dist_disp_wifi = pd.DataFrame({
    'dispositivo': [c.replace('Dispositivo_conexión_', '').replace('_', ' ') for c in cols_dispositivo],
    'total_usuarios': [total_disp[c] for c in cols_dispositivo],
    'pct': [(total_disp[c] / total_disp.sum() * 100).round(2) for c in cols_dispositivo]
})
dist_disp_wifi.to_csv(f'{OUTPUT_DIR}/distribucion_dispositivo_wifi.csv', index=False)

# ── 7. Tendencia temporal ────────────────────────────────────────────────────
if 'tend_dia' in dir():
    tend_dia.to_csv(f'{OUTPUT_DIR}/tendencia_anual_wifi.csv', index=False)

# ── 8. Master consolidado barrio (WiFi + segmentación) ──────────────────────
master_wifi = flujo_barrio.merge(
    perfil_barrio[['barrio_norm', 'total_usuarios', 'rango_edad_dominante',
                   'dispositivo_dominante', 'ratio_smartphone_pc', 'perfil_consumidor']],
    on='barrio_norm', how='left'
)
master_wifi.to_csv(f'{OUTPUT_DIR}/master_wifi_barrio.csv', index=False)

print('✅ Archivos exportados:')
for f in sorted(os.listdir(OUTPUT_DIR)):
    fsize = os.path.getsize(f'{OUTPUT_DIR}/{f}')
    print(f'   {f}  ({fsize/1024:.1f} KB)')

✅ Archivos exportados:


   crosstab_tipo_barrio.csv  (2.4 KB)
   densidad_emprendedora_barrio.csv  (3.5 KB)
   densidad_emprendedora_comuna.csv  (0.6 KB)
   distribucion_dispositivo_wifi.csv  (0.1 KB)
   distribucion_edad_emprendedores.csv  (0.1 KB)
   distribucion_edad_wifi.csv  (0.2 KB)
   flujo_wifi_barrio.csv  (1.0 KB)
   flujo_wifi_comuna.csv  (0.9 KB)
   flujo_wifi_sede.csv  (36.2 KB)
   indice_diversificacion_barrio.csv  (4.8 KB)
   master_emprendedores_barrio.csv  (7.6 KB)
   master_wifi_barrio.csv  (2.2 KB)
   perfil_consumidor_wifi_barrio.csv  (4.1 KB)
   perfil_demografico_barrio.csv  (5.7 KB)
   perfil_demografico_zona.csv  (0.4 KB)
   tendencia_anual_wifi.csv  (0.1 KB)
   tipo_dominante_barrio.csv  (2.6 KB)


In [12]:
# ── Resumen estadístico final ────────────────────────────────────────────────
print('=' * 60)
print('RESUMEN ESTADÍSTICO FINAL - WiFi')
print('=' * 60)
print(f'Sedes WiFi analizadas: {len(flujo_sede)}')
print(f'Barrios con cobertura WiFi: {len(flujo_barrio)}')
print(f'Total registros históricos: {flujo_barrio["total_registros"].sum():,.0f}')
print()
top_barrio = flujo_barrio.iloc[0]
print(f'Barrio con más tráfico: {top_barrio["barrio_norm"]} ({top_barrio["total_registros"]:,.0f} registros)')
top_sede = flujo_sede.sort_values('total_registros', ascending=False).iloc[0]
print(f'Sede con más tráfico: {top_sede["sede_norm"]} ({top_sede["total_registros"]:,.0f} registros)')
print()
print(f'Rango de edad dominante global: {dist_edad_wifi.sort_values("total_usuarios", ascending=False).iloc[0]["rango_edad"]}')
print(f'Dispositivo dominante global: {dist_disp_wifi.sort_values("total_usuarios", ascending=False).iloc[0]["dispositivo"]}')
print()
print(f'Barrios perfil digital-movil: {(perfil_barrio["perfil_consumidor"] == "digital-movil").sum()}')
print(f'Barrios perfil mixto-digital: {(perfil_barrio["perfil_consumidor"] == "mixto-digital").sum()}')
print(f'Barrios perfil pc-tradicional: {(perfil_barrio["perfil_consumidor"] == "pc-tradicional").sum()}')

RESUMEN ESTADÍSTICO FINAL - WiFi
Sedes WiFi analizadas: 393
Barrios con cobertura WiFi: 21
Total registros históricos: 7,933,913

Barrio con más tráfico: la candelaria (931,607 registros)
Sede con más tráfico: terminal del sur (236,797 registros)

Rango de edad dominante global: De 18 a 28 Años
Dispositivo dominante global: Smartphone

Barrios perfil digital-movil: 21
Barrios perfil mixto-digital: 0
Barrios perfil pc-tradicional: 0
